In [1]:
import os
import asyncio
import aiohttp
import pandas as pd
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()

In [2]:
from dotenv import load_dotenv
load_dotenv()
os.makedirs('../data/raw', exist_ok=True)

### Request data using API

In [3]:
URL = "https://data.nayaone.com/banksim"
BATCH_SIZE = 10  # API rate limit per request
limit_size = None  # max rows (no limit = None)
OUTPUT_PATH = '../data/raw/banksim.csv'
HEADERS = {
    'Accept-Profile': 'api',
    'sandpit-key': os.getenv('API_KEY')
}

In [4]:
import requests
def find_last_offset(url):
    offset = 100000
    data = True
    while data:
        data = requests.get(url, headers=HEADERS, params={'offset': offset}).json()
        print(f"offset: {offset}")
        offset *= 2

    left, right = 0, offset - 1
    answer = -1

    while left <= right:
        mid = (left + right) // 2
        value = requests.get(url, headers=HEADERS, params={'offset': mid}).json()

        if value:
            answer = mid
            left = mid + 1   # search right half
        else:
            right = mid - 1  # search left half
        print(f"left: {left} right: {right}")
    return answer+1

In [5]:
limit = find_last_offset(URL)
print(limit)

offset: 100000
offset: 200000
offset: 400000
offset: 800000
left: 0 right: 799998
left: 400000 right: 799998
left: 400000 right: 599998
left: 500000 right: 599998
left: 550000 right: 599998
left: 575000 right: 599998
left: 587500 right: 599998
left: 593750 right: 599998
left: 593750 right: 596873
left: 593750 right: 595310
left: 594531 right: 595310
left: 594531 right: 594919
left: 594531 right: 594724
left: 594628 right: 594724
left: 594628 right: 594675
left: 594628 right: 594650
left: 594640 right: 594650
left: 594640 right: 594644
left: 594640 right: 594641
left: 594641 right: 594641
left: 594642 right: 594641
594642


In [6]:
if limit_size is None:
    LIMIT_SIZE = limit
else:
    LIMIT_SIZE = limit_size

Async parallel fetch function

In [7]:
async def fetch_page(session, offset):
    async with session.get(URL, headers=HEADERS, params={'offset': offset}) as resp:
        return offset, await resp.json()

Main parallel fetch loop

In [8]:
CONCURRENCY = 50

In [9]:
async def fetch_all():
    results = {}
    pbar = tqdm(total=LIMIT_SIZE, unit=" rows", desc="Fetching Data")

    connector = aiohttp.TCPConnector(
        limit=CONCURRENCY,
        ttl_dns_cache=6000,
        enable_cleanup_closed=True
    )

    sem = asyncio.Semaphore(CONCURRENCY)

    async def fetch_limited(offset):
        async with sem:
            return await fetch_page(session, offset)

    async with aiohttp.ClientSession(connector=connector) as session:
        offsets = range(0, LIMIT_SIZE, BATCH_SIZE)
        tasks = [asyncio.create_task(fetch_limited(o)) for o in offsets]

        for fut in asyncio.as_completed(tasks):
            off, data = await fut
            if not data:
                break

            results[off] = data
            pbar.update(len(data))

    pbar.close()
    return results

In [10]:
# Run async fetcher
results = asyncio.run(fetch_all())

Fetching Data:   7%|▋         | 42090/594642 [02:40<35:05, 262.44 rows/s]  


KeyboardInterrupt: 

In [ ]:
# Flatten results in correct order
ordered_offsets = sorted(results.keys())
all_rows = [row for off in ordered_offsets for row in results[off]]

df = pd.DataFrame(all_rows)
df.to_csv(OUTPUT_PATH, index=False)

print(f"{len(all_rows)} rows saved to {OUTPUT_PATH}")

1790 rows saved to ../data/raw/banksim.csv
